# Práctica 2: Reconocimiento Activo
## Descubrimiento de hosts con Scapy

**Autor:** Marcos Pérez
**Fecha:** 17/04/2026

Este notebook implementa descubrimiento de hosts activos usando:
- ICMP Timestamp Request (tipo 13)
- TCP ACK (puerto configurable)
- UDP (puerto configurable)

## 1. Importar librerías necesarias

In [ ]:
from scapy.all import *
import random
import time

print("Librerías importadas correctamente")

## 2. Función craft_discovery_pkts

Construye paquetes según los protocolos especificados.

In [ ]:
def craft_discovery_pkts(protocolos, ip_range, puerto=80, num_paquetes=None):
    """
    Construye paquetes para descubrimiento de hosts activos
    """
    paquetes = []
    
    if isinstance(protocolos, str):
        protocolos = [protocolos]
    
    if len(protocolos) > 3:
        protocolos = protocolos[:3]
    
    if num_paquetes is None:
        num_paquetes = {proto: 1 for proto in protocolos}
    
    for proto in protocolos:
        for _ in range(num_paquetes.get(proto, 1)):
            if proto.upper() == "UDP":
                pkt = IP(dst=ip_range)/UDP(dport=puerto)/Raw(load="DISCOVERY")
                paquetes.append(pkt)
            elif proto.upper() == "TCP":
                pkt = IP(dst=ip_range)/TCP(dport=puerto, flags="A")
                paquetes.append(pkt)
            elif proto.upper() == "ICMP":
                pkt = IP(dst=ip_range)/ICMP(type=13, code=0)
                paquetes.append(pkt)
    
    return paquetes

print("Función definida correctamente")

## 3. Prueba de construcción de paquetes

In [ ]:
# Probar construcción de paquetes
print("=== Construcción de paquetes ===")

# ICMP
pkt_icmp = craft_discovery_pkts("ICMP", "172.20.0.10")
print(f"ICMP: {pkt_icmp[0].summary()}")

# TCP
pkt_tcp = craft_discovery_pkts("TCP", "172.20.0.10", puerto=80)
print(f"TCP: {pkt_tcp[0].summary()}")

# UDP
pkt_udp = craft_discovery_pkts("UDP", "172.20.0.10", puerto=53)
print(f"UDP: {pkt_udp[0].summary()}")

# Múltiples protocolos
pkt_multi = craft_discovery_pkts(["ICMP", "TCP"], "172.20.0.10")
print(f"Múltiples: {len(pkt_multi)} paquetes construidos")

## 4. Envío de paquetes y recepción de respuestas

In [ ]:
# Probar envío a host activo
print("=== Envío a host activo (172.20.0.10) ===")

# ICMP
pkt = craft_discovery_pkts("ICMP", "172.20.0.10")[0]
respuesta = sr1(pkt, timeout=2, verbose=False)

if respuesta:
    print(f"✓ ICMP: Respuesta recibida de {respuesta.src}")
    if ICMP in respuesta:
        print(f"  Tipo ICMP: {respuesta[ICMP].type}")
else:
    print("✗ ICMP: Sin respuesta")

# TCP
pkt = craft_discovery_pkts("TCP", "172.20.0.10", puerto=80)[0]
respuesta = sr1(pkt, timeout=2, verbose=False)

if respuesta:
    print(f"✓ TCP: Respuesta recibida de {respuesta.src}")
    if TCP in respuesta:
        print(f"  Flags TCP: {respuesta[TCP].flags}")
else:
    print("✗ TCP: Sin respuesta")

## 5. Escaneo de rango completo

In [ ]:
def descubrir_hosts(ip_range, protocolos=["ICMP", "TCP"], puerto=80):
    """
    Descubre hosts activos en un rango de IPs
    """
    hosts_activos = set()
    
    for proto in protocolos:
        print(f"\n[*] Probando con {proto}...")
        paquetes = craft_discovery_pkts(proto, ip_range, puerto)
        
        if proto == "UDP":
            respuestas, _ = sr(paquetes, timeout=3, verbose=False)
        else:
            respuestas, _ = sr(paquetes, timeout=2, verbose=False)
        
        for respuesta in respuestas:
            ip = respuesta[1].src
            hosts_activos.add(ip)
            print(f"  [+] Host activo: {ip}")
    
    return list(hosts_activos)

# Ejecutar escaneo
print("=== ESCANEO DE RED 172.20.0.0/24 ===\n")
hosts = descubrir_hosts("172.20.0.0/24", ["ICMP", "TCP"], 80)
print(f"\n=== RESULTADOS ===")
print(f"Hosts activos encontrados: {len(hosts)}")
for i, host in enumerate(sorted(hosts), 1):
    print(f"  {i}. {host}")

## 6. Prueba con IP inactiva

In [ ]:
# Probar con IP que no tiene host
print("=== PRUEBA CON IP INACTIVA (172.20.0.99) ===\n")

pkt = craft_discovery_pkts("ICMP", "172.20.0.99")[0]
respuesta = sr1(pkt, timeout=2, verbose=False)

if respuesta:
    print(f"Respuesta recibida de {respuesta.src}")
else:
    print("✓ Sin respuesta - IP correctamente identificada como inactiva")